# Quantum Computing: Variational Quantum Eigensolver (VQE)

## Hybrid Quantum-Classical Algorithm for Ground State Energy

**2025 Nobel Prize**: Clarke, Devoret, Martinis for macroscopic quantum tunneling (superconducting qubits)  
**Problem**: Find ground state energy E₀ = min ⟨ψ|H|ψ⟩  
**Challenge**: Exponential Hilbert space (2^n states)  
**Solution**: Variational principle + quantum measurement + classical optimization


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import eigh

np.random.seed(42)

## Part 1: Theory

### Variational Principle
For any state |ψ⟩: ⟨ψ|H|ψ⟩ ≥ E₀ (ground state energy)

### Parametrized Circuit (Ansatz)
|ψ(θ)⟩ = U(θ)|0⟩

### VQE Algorithm
```
1. Measure energy E(θ) on quantum device
2. θ_{t+1} = θ_t - η ∇E(θ_t)  [classical optimization]
3. Repeat until convergence
```

### Barren Plateau Problem
Random initialization → ∇E ≈ 0 exponentially (vanishing gradient)  
**Solution**: Problem-inspired ansatz, warm-starting


## Part 2: Training Dynamics


In [ ]:
# Simulate VQE optimization trajectory
class VQETrajectory:
    def __init__(self, vqe):
        self.vqe = vqe
        self.E_history = []
        self.call_count = 0
    
    def energy_with_history(self, theta):
        E = self.vqe.energy(theta)
        self.E_history.append(E)
        self.call_count += 1
        return E
    
    def optimize(self):
        theta0 = np.random.randn(4) * 0.5
        result = minimize(self.energy_with_history, theta0, method='COBYLA')
        return np.array(self.E_history)

trajectory = VQETrajectory(vqe)
energies = trajectory.optimize()

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Energy convergence
ax = axes[0]
ax.plot(energies, 'o-', linewidth=2, markersize=4, color='steelblue')
ax.axhline(y=vqe.E0, color='red', linestyle='--', linewidth=2, label=f'True E₀')
ax.set_xlabel('Optimization Iteration')
ax.set_ylabel('Energy')
ax.set_title('VQE Convergence: Energy vs Iteration')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Error decay
ax = axes[1]
errors = np.abs(energies - vqe.E0)
ax.semilogy(errors, 'o-', linewidth=2, markersize=4, color='darkgreen')
ax.set_xlabel('Optimization Iteration')
ax.set_ylabel('|E - E₀| (log scale)')
ax.set_title('Error Decay')
ax.grid(True, alpha=0.3, which='both')

# Plot 3: Barren plateau illustration
ax = axes[2]
theta_range = np.linspace(-np.pi, np.pi, 50)
energies_random = []
for t in theta_range:
    theta_test = np.random.randn(4) * 0.01
    theta_test[0] = t
    energies_random.append(vqe.energy(theta_test))

ax.plot(theta_range, energies_random, 'o-', linewidth=2, color='orange')
ax.axhline(y=np.mean(energies_random), color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Parameter θ')
ax.set_ylabel('Energy')
ax.set_title('Barren Plateau: Flat Loss Landscape (random init)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('SECTION_4_QUANTUM/vqe.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nConvergence: Energy decreased from {energies[0]:.4f} to {energies[-1]:.4f}")
print(f"Final error: {abs(energies[-1] - vqe.E0):.6f}")

## Key Insights

1. **Hybrid approach**: Quantum for exponential sampling, classical for optimization
2. **NISQ era**: Works with noisy shallow circuits (50-500 gates)
3. **Barren plateaus**: Avoid random initialization; use problem-inspired ansatz
4. **Hardware constraints**: Limited by circuit depth, coherence time (~100 μs), gate fidelity (~99.9%)
5. **Applications**: Quantum chemistry, optimization, materials science

### References
- Peruzzo, A., et al. (2014). "A variational eigenvalue solver on a photonic quantum processor"
- 2025 Nobel Prize in Physics: Clarke, Devoret, Martinis
